# Silver: Change Data Capture Detection

Detects changes (INSERT, UPDATE, DELETE) between staging and current silver tables.

## Logic

1. **INSERT**: New jobs in staging not in current
2. **UPDATE**: Jobs in both with different record_hash
3. **DELETE**: Jobs in current not in staging (set is_active=False)
4. **RESTORE**: Previously deleted jobs reappearing

## Outputs

- Updates `silver.silver_jobs_current` with changes
- Logs changes to `silver.silver_job_changes`
- Idempotent: safe to re-run

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for latest staging)")
dbutils.widgets.dropdown("enable_deletes", "true", ["true", "false"], "Enable soft deletes")

batch_id = dbutils.widgets.get("batch_id").strip()
enable_deletes = dbutils.widgets.get("enable_deletes") == "true"

print(f"Parameters:")
print(f"  Batch ID: {batch_id if batch_id else 'Latest staging'}")
print(f"  Enable Deletes: {enable_deletes}")

In [0]:
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = F.current_timestamp()

print(f"Run ID: {run_id}")

In [0]:
# Load staging data
staging_df = spark.table(f"{SILVER_SCHEMA}.silver_jobs_staging")
staging_count = staging_df.count()
print(f"Staging records: {staging_count}")

# Load current silver data (create if doesn't exist)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_jobs_current (
  enterprise_job_id STRING NOT NULL,
  source_name STRING,
  source_job_id STRING,
  source_job_key STRING,
  company_name_raw STRING,
  company_name_norm STRING,
  title_raw STRING,
  title_normalized STRING,
  description_raw STRING,
  location_raw STRING,
  location_norm STRING,
  remote_type STRING,
  source_url STRING,
  posted_at TIMESTAMP,
  last_seen TIMESTAMP,
  is_active BOOLEAN,
  soft_delete_flag BOOLEAN,
  soft_delete_reason STRING,
  record_hash STRING,
  current_batch_id STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
""")

current_df = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current")
current_count = current_df.count()
print(f"Current records: {current_count}")

In [0]:
# New jobs (not in current)
inserts_df = staging_df.alias("stg").join(
    current_df.alias("cur"),
    F.col("stg.source_job_key") == F.col("cur.source_job_key"),
    "left_anti"
).select(
    F.expr("uuid()").alias("enterprise_job_id"),
    F.col("source_name"),
    F.col("source_job_id"),
    F.col("source_job_key"),
    F.col("company_name_raw"),
    F.col("company_name_norm"),
    F.col("title_raw"),
    F.col("title_normalized"),
    F.col("description_raw"),
    F.col("location_raw"),
    F.col("location_norm"),
    F.col("remote_type"),
    F.col("source_url"),
    F.col("posted_at"),
    F.col("last_seen"),
    F.lit(True).alias("is_active"),
    F.lit(False).alias("soft_delete_flag"),
    F.lit(None).cast("string").alias("soft_delete_reason"),
    F.col("record_hash"),
    F.col("batch_id").alias("current_batch_id"),
    run_timestamp.alias("created_at"),
    run_timestamp.alias("updated_at")
).withColumn("change_type", F.lit("INSERT"))

insert_count = inserts_df.count()
print(f"New jobs to INSERT: {insert_count}")

In [0]:
# Updated jobs (in both, different hash)
updates_df = staging_df.alias("stg").join(
    current_df.alias("cur"),
    F.col("stg.source_job_key") == F.col("cur.source_job_key"),
    "inner"
).where(
    F.col("stg.record_hash") != F.col("cur.record_hash")
).select(
    F.col("cur.enterprise_job_id"),
    F.col("stg.source_name"),
    F.col("stg.source_job_id"),
    F.col("stg.source_job_key"),
    F.col("stg.company_name_raw"),
    F.col("stg.company_name_norm"),
    F.col("stg.title_raw"),
    F.col("stg.title_normalized"),
    F.col("stg.description_raw"),
    F.col("stg.location_raw"),
    F.col("stg.location_norm"),
    F.col("stg.remote_type"),
    F.col("stg.source_url"),
    F.col("stg.posted_at"),
    F.col("stg.last_seen"),
    F.lit(True).alias("is_active"),
    F.lit(False).alias("soft_delete_flag"),
    F.lit(None).cast("string").alias("soft_delete_reason"),
    F.col("stg.record_hash"),
    F.col("stg.batch_id").alias("current_batch_id"),
    run_timestamp.alias("created_at"),
    run_timestamp.alias("updated_at")
).withColumn("change_type", F.lit("UPDATE"))

update_count = updates_df.count()
print(f"Jobs to UPDATE: {update_count}")

In [0]:
# Deleted jobs (in current, not in staging, currently active)
if enable_deletes:
    deletes_df = current_df.alias("cur").join(
        staging_df.alias("stg"),
        F.col("cur.source_job_key") == F.col("stg.source_job_key"),
        "left_anti"
    ).where(
        F.col("cur.is_active") == True
    ).select(
        F.col("cur.*")
    ).withColumn("is_active", F.lit(False)) \
     .withColumn("soft_delete_flag", F.lit(True)) \
     .withColumn("soft_delete_reason", F.lit("Not seen in latest source snapshot")) \
     .withColumn("updated_at", run_timestamp) \
     .withColumn("change_type", F.lit("DELETE"))
    
    delete_count = deletes_df.count()
    print(f"Jobs to soft DELETE: {delete_count}")
else:
    deletes_df = spark.createDataFrame([], current_df.schema).withColumn("change_type", F.lit("DELETE"))
    delete_count = 0
    print("Soft deletes disabled")

In [0]:
# Combine all changes - align schemas first
common_columns = [
    "enterprise_job_id", "source_name", "source_job_id", "source_job_key",
    "company_name_raw", "company_name_norm", "title_raw", "title_normalized",
    "description_raw", "location_raw", "location_norm", "remote_type",
    "source_url", "posted_at", "last_seen", "is_active",
    "soft_delete_flag", "soft_delete_reason", "record_hash", "current_batch_id",
    "created_at", "updated_at"
]

# Normalize deletes_df column names and add missing columns
if "created_at" in deletes_df.columns:
    deletes_aligned = deletes_df \
        .withColumnRenamed("company_name", "company_name_raw") \
        .withColumn("source_url", F.lit(None).cast("string")) \
        .select(*common_columns)
else:
    deletes_aligned = deletes_df \
        .withColumnRenamed("company_name", "company_name_raw") \
        .withColumn("source_url", F.lit(None).cast("string")) \
        .withColumn("created_at", run_timestamp) \
        .select(*common_columns)

all_changes_df = inserts_df.select(*common_columns) \
    .union(updates_df.select(*common_columns)) \
    .union(deletes_aligned)

# Write to current table using merge
from delta.tables import DeltaTable

if current_count == 0:
    # First load - just write
    all_changes_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.silver_jobs_current")
    print("Initial load complete")
else:
    # Merge logic
    delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
    
    delta_current.alias("cur").merge(
        all_changes_df.alias("chg"),
        "cur.enterprise_job_id = chg.enterprise_job_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print("Merge complete")

final_count = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current").count()
print(f"Final record count: {final_count}")

In [0]:
# Create changes audit table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_job_changes (
  change_id STRING,
  enterprise_job_id STRING,
  source_name STRING,
  change_type STRING,
  changed_columns ARRAY<STRING>,
  old_value_json STRING,
  new_value_json STRING,
  change_timestamp TIMESTAMP,
  batch_id STRING
)
USING DELTA
""")

# Log all changes
inserts_log = inserts_df.select(
    F.expr("uuid()").alias("change_id"),
    F.col("enterprise_job_id"),
    F.col("source_name"),
    F.lit("INSERT").alias("change_type"),
    F.lit(None).cast("array<string>").alias("changed_columns"),
    F.lit(None).cast("string").alias("old_value_json"),
    F.lit(None).cast("string").alias("new_value_json"),
    run_timestamp.alias("change_timestamp"),
    F.col("current_batch_id").alias("batch_id")
)

updates_log = updates_df.select(
    F.expr("uuid()").alias("change_id"),
    F.col("enterprise_job_id"),
    F.col("source_name"),
    F.lit("UPDATE").alias("change_type"),
    F.lit(None).cast("array<string>").alias("changed_columns"),
    F.lit(None).cast("string").alias("old_value_json"),
    F.lit(None).cast("string").alias("new_value_json"),
    run_timestamp.alias("change_timestamp"),
    F.col("current_batch_id").alias("batch_id")
)

# Handle potential schema mismatch if table was overwritten
try:
    deletes_log = deletes_df.select(
        F.expr("uuid()").alias("change_id"),
        F.col("enterprise_job_id"),
        F.col("source_name"),
        F.lit("DELETE").alias("change_type"),
        F.lit(None).cast("array<string>").alias("changed_columns"),
        F.lit(None).cast("string").alias("old_value_json"),
        F.lit(None).cast("string").alias("new_value_json"),
        run_timestamp.alias("change_timestamp"),
        F.col("current_batch_id").alias("batch_id")
    )
except Exception:
    # Schema mismatch - create empty deletes_log
    deletes_log = spark.createDataFrame([], inserts_log.schema)

all_logs = inserts_log.union(updates_log).union(deletes_log)
all_logs.write.format("delta").mode("append").saveAsTable(f"{SILVER_SCHEMA}.silver_job_changes")

print(f"Logged {all_logs.count()} changes to audit table")

In [0]:
result = {
    "status": "success",
    "run_id": run_id,
    "inserts": insert_count,
    "updates": update_count,
    "deletes": delete_count,
    "total_changes": insert_count + update_count + delete_count,
    "final_count": final_count
}

print("\n=== CDC Summary ===")
print(json.dumps(result, indent=2))

dbutils.notebook.exit(json.dumps(result))